In [1]:
# ============ CONCAT DE: clean DE + EN translation ============
import os, re, json
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, MarianMTModel, MarianTokenizer
from datasets import load_dataset, Dataset

def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c): return c
CACHE_DIR = find_cache_dir()
RESULTS_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'results')

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
print(f"Device: {device}")

collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))

# E5
e5_name = 'intfloat/multilingual-e5-large'
tok = AutoTokenizer.from_pretrained(e5_name)
model = AutoModel.from_pretrained(e5_name).to(device).eval()

# DE -> EN translator
mt_name = "Helsinki-NLP/opus-mt-de-en"
mt_tok = MarianTokenizer.from_pretrained(mt_name)
mt_model = MarianMTModel.from_pretrained(mt_name).to(device).eval()
print("Models loaded.")

def avg_pool(h, m):
    h = h.masked_fill(~m[..., None].bool(), 0.0)
    return h.sum(dim=1) / m.sum(dim=1)[..., None]

def clean_tweet(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    t = re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def translate(texts, bs=32):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc="Translate"):
        b = texts[i:i+bs]
        ins = mt_tok(b, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        with torch.no_grad():
            g = mt_model.generate(**ins, max_length=256, num_beams=1)
        out.extend(mt_tok.batch_decode(g, skip_special_tokens=True))
    return out

def embed(texts, max_len=160):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    ds = ds.map(lambda e: tok(e["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt"),
                batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for b in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embed"):
            b = {k: v.to(device) for k, v in b.items()}
            embs.append(avg_pool(model(**b).last_hidden_state, b['attention_mask']).cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def mrr5(cs, lab):
    top5 = np.argsort(-cs, axis=1)[:, :5]
    keys = collection_keys[top5]
    m = (keys == lab[:, None])
    pos = np.argmax(m, axis=1); pos[~m.any(axis=1)] = -1
    rr = np.zeros_like(pos, dtype=float); rr[pos >= 0] = 1.0 / (pos[pos >= 0] + 1)
    return float(rr.mean())

# DE data
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "de")
dev_raw = data["dev"]
dev_labels = np.array(dev_raw["pubkey"])

dev_clean = [clean_tweet(t) for t in dev_raw["text"]]
print(f"\nTranslating {len(dev_clean)} DE dev tweets...")
dev_en = translate(dev_clean)

# Save translations
with open(os.path.join(CACHE_DIR, 'dev_en_de.json'), 'w', encoding='utf-8') as f:
    json.dump(dev_en, f, ensure_ascii=False)

# Concat + embed
dev_concat = [f"{clean} {en}" for clean, en in zip(dev_clean, dev_en)]
for i in range(3):
    print(f"[{i}] {dev_concat[i][:200]}\n")

emb = embed(dev_concat)
score = mrr5((emb @ collection_embeddings.T).numpy(), dev_labels)

print(f"\n=== DE DEV — concat (clean DE + EN trans) ===")
print(f"MRR@5: {score:.4f}")
print(f"Baseline DE: 0.3767, Cleaning DE: 0.3803")
print(f"Δ vs cleaning: {score - 0.3803:+.4f}")

torch.save(emb, os.path.join(CACHE_DIR, 'dev_embeddings_de_concat.pt'))
log = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp9b_concat_de_dev",
    "transformation": "clean_de + en_translation concat",
    "language": "de",
    "mrr5_dev": score,
    "delta_vs_cleaning": score - 0.3803,
}
with open(os.path.join(RESULTS_DIR, 'results.jsonl'), 'a') as f:
    f.write(json.dumps(log) + '\n')
print("Logged.")

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8182.15it/s]
/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 2190.45it/s]


Models loaded.

Translating 386 DE dev tweets...


Translate: 100%|██████████| 13/13 [00:34<00:00,  2.62s/it]


[0] Die Infektion kann symptomfrei ablaufen oder sich wie eine Erkältung zeigen. Bei manchen Krankheitsbetroffenen kommt es nach ungefähr einer Woche zu den bangen schweren Krankheitsverläufen. CoronaViru

[1] Die Rechenzentrumsindustrie fußt auf einer linearen Wirtschaft. Allerdings besteht ein akuter Handlungsbedarf, zu einer CircularEconomy zu wechseln. Wie das aussehen kann welche Herausforderungen sich

[2] Das Forscherteam von Prof. Dr. , Institut für Virologie der , hat in Zusammenarbeit mit WissenschaftlerInnen aus Großbritannien unter Einsatz künstlicher Intelligenz 2 Substanzen als potenzielle COVID



Embed: 100%|██████████| 25/25 [00:23<00:00,  1.05it/s]



=== DE DEV — concat (clean DE + EN trans) ===
MRR@5: 0.3861
Baseline DE: 0.3767, Cleaning DE: 0.3803
Δ vs cleaning: +0.0058
Logged.
